In [8]:
!pip install -q langchain-groq

In [9]:
import os, requests
from google.colab import userdata
from langchain_groq import ChatGroq
from pathlib import Path
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import PromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.prompts.chat import HumanMessagePromptTemplate, SystemMessagePromptTemplate, ChatPromptTemplate, AIMessagePromptTemplate

below cell is just for the API key , mene kiunke use kya hai Colab server , to mjhe key ko us server se hi uthana tha , islye yeh sb generate kya , chalane se pehly agr colab UI nhi (vs code wagera hai), to pehly file ko upload krdo colab server pr

In [10]:

def _read_env_file(name, filename="secrets.env"):
    """Look for KEY=value in secrets.env in the working dir or /content."""
    for folder in (Path.cwd(), Path("/content")):
        path = folder / filename
        if path.is_file():
            for line in path.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, _, val = line.partition("=")
                if key.strip().removeprefix("export ").strip() == name:
                    return val.strip().strip('"').strip("'")
    return None

def get_secret(name, required=True):
    value = None
    try:
        value = userdata.get(name)      # works in the Colab web UI
    except Exception:
        pass                            # timeout / not found / no access
    if not value:
        value = _read_env_file(name)    # fallback: uploaded secrets.env
    if value:
        os.environ[name] = value
    elif required:
        raise RuntimeError(
            f"{name} not found in Colab Secrets or secrets.env. "
            "Upload secrets.env to the Colab server and re-run."
        )
    return value

groq_key = get_secret("GROQ_API_KEY")


print("GROQ key loaded:", bool(groq_key))


GROQ key loaded: True


To view models available

In [11]:
r = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"},
)
for m in r.json()["data"]:
    print(m["id"])

allam-2-7b
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-20b
openai/gpt-oss-120b
whisper-large-v3
whisper-large-v3-turbo
qwen/qwen3.8-27b
canopylabs/orpheus-v1-english
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-22m


In [12]:
llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    model_kwargs= {"seed" : 350 },
    temperature= 0,
    max_tokens= 250
)

response = llm.invoke("Very warm day")

print(response)

content='Sounds like the sun’s really turning up the heat! 🌞\n\nHere are a few ideas to stay comfortable (and maybe even enjoy the warmth) on a hot day:\n\n**Quick cooling tricks**\n- **Hydrate**: Keep a water bottle handy and sip regularly. Adding a few slices of cucumber, lemon, or mint can make it extra refreshing.\n- **Cold compress**: Slip a cool, damp washcloth over your neck, wrists, or forehead for an instant chill.\n- **Ice‑cold snack**: Popsicles, frozen fruit, or a bowl of chilled yogurt can help lower your core temperature from the inside out.\n\n**Indoor comfort**\n- **Shade & curtains**: Close blinds or curtains on the sun‑lit side of the house to keep the indoor temperature down.\n- **Fans + ice**: Place a shallow pan of ice or frozen water bottles in front of a fan—air blowing over the ice creates a' additional_kwargs={'reasoning_content': 'The user says "Very warm day". Likely they are making a statement, maybe expecting a response. As ChatGPT, we can respond with empa

In [13]:
print(response.content)

Sounds like the sun’s really turning up the heat! 🌞

Here are a few ideas to stay comfortable (and maybe even enjoy the warmth) on a hot day:

**Quick cooling tricks**
- **Hydrate**: Keep a water bottle handy and sip regularly. Adding a few slices of cucumber, lemon, or mint can make it extra refreshing.
- **Cold compress**: Slip a cool, damp washcloth over your neck, wrists, or forehead for an instant chill.
- **Ice‑cold snack**: Popsicles, frozen fruit, or a bowl of chilled yogurt can help lower your core temperature from the inside out.

**Indoor comfort**
- **Shade & curtains**: Close blinds or curtains on the sun‑lit side of the house to keep the indoor temperature down.
- **Fans + ice**: Place a shallow pan of ice or frozen water bottles in front of a fan—air blowing over the ice creates a


In [14]:
messages = [
    HumanMessage(content="Sometimes it feels like the challenges in our world are too big to solve. Is there any reason to stay optimistic about the future?"),
    AIMessage(content="It is completely natural to feel overwhelmed, but there is immense reason for hope. Every day, brilliant minds are developing clean energy technologies, medical breakthroughs are curing previously untreatable diseases, and communities are coming together to support one another. Progress might seem slow, but humanity's capacity for resilience and innovation has always guided us through our darkest times."),
    HumanMessage(content="That makes me feel a bit better. How can an individual contribute to making things brighter?"),
    AIMessage(content="Every grand forest begins with a single small seed. By spreading kindness, continuing to learn, and making small sustainable choices daily, you contribute to a powerful ripple effect of positive change. Never underestimate the impact of your hope and actions!"),
    HumanMessage(content = "Feeling sad")
]

response = llm.invoke(messages)
print(response.content)

I’m really sorry you’re feeling this way. It can be tough when sadness settles in, but you don’t have to sit with it alone.

**A few gentle steps that might help right now**

| What to try | Why it can help |
|-------------|-----------------|
| **Name the feeling** – say to yourself, “I’m feeling sad,” or write it down. | Labeling emotions can reduce their intensity and give you a little distance. |
| **Ground yourself** – focus on five things you can see, four you can touch, three you can hear, two you can smell, one you can taste. | This simple “5‑4‑3‑2‑1” exercise pulls your attention away from rumination and into the present moment. |
| **Move a little** – stretch, walk around the room, or do a short dance to a favorite song. | Light physical activity releases endorphins


In [15]:
Template = """
System : {description}

Human : I recently purchased a {car name}, give me manufacturer name for {car name}
"""

In [16]:
template_prompt = PromptTemplate.from_template(template= Template)

prompt_value = template_prompt.invoke({"description": "chatbot should reply in a car dealer manner", "car name": "Civic"})

In [17]:
prompt_value

StringPromptValue(text='\nSystem : chatbot should reply in a car dealer manner\n\nHuman : I recently purchased a Civic, give me manufacturer name for Civic\n')

In [18]:
print(prompt_value.text)


System : chatbot should reply in a car dealer manner

Human : I recently purchased a Civic, give me manufacturer name for Civic



In [19]:
message_sys = "{description}",
message_h = "I recently purchased a {car name}, give me manufacturer name for {car name} "

system_msg = SystemMessagePromptTemplate.from_template(message_sys)
human_msg = HumanMessagePromptTemplate.from_template(message_h)

In [20]:
chat_template = ChatPromptTemplate.from_messages([system_msg, human_msg])

In [21]:
chat_values = chat_template.invoke({"description": "chatbot should reply in a car dealer manner", "car name": "Civic"})

In [22]:
response = llm.invoke(chat_values)

In [23]:
print(response.content)

Absolutely—great choice! The Honda Civic is built by **Honda Motor Co., Ltd.**. If you ever want to explore accessories, service plans, or even a future upgrade, just let us know—we’re here to keep you cruising in style! 🚗💨


**Few shot prompting (means example derhe hain )**

In [24]:
template_h = "Im watching {movie_name}, how is it ?"
template_Ai = "{review}"

In [25]:
human_template = HumanMessagePromptTemplate.from_template(template= template_h)
Ai_template = AIMessagePromptTemplate.from_template(template= template_Ai)

In [26]:
chat_template = ChatPromptTemplate.from_messages([human_template, Ai_template])

In [27]:
examples = [
    {
        "movie_name": "Inception",
        "review": "An mind-bending sci-fi thriller that beautifully explores the subconscious layers of dreams."
    },
    {
        "movie_name": "The Dark Knight",
        "review": "A masterful superhero film defined by its gritty realism and an unforgettable performance by Heath Ledger."
    },
    {
        "movie_name": "Interstellar",
        "review": "A visually stunning and emotionally powerful space odyssey about humanity's survival and love across dimensions."
    }
]

In [29]:
few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples = examples,
    example_prompt= chat_template,
    input_variables = ["movie_name"]
)

In [30]:
main_chat_template = ChatPromptTemplate.from_messages([few_shot_prompt, human_template])

In [38]:
chat_values = main_chat_template.invoke({"movie_name" : "joker"})

In [39]:
chat_values

ChatPromptValue(messages=[HumanMessage(content='Im watching Inception, how is it ?', additional_kwargs={}, response_metadata={}), AIMessage(content='An mind-bending sci-fi thriller that beautifully explores the subconscious layers of dreams.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Im watching The Dark Knight, how is it ?', additional_kwargs={}, response_metadata={}), AIMessage(content='A masterful superhero film defined by its gritty realism and an unforgettable performance by Heath Ledger.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Im watching Interstellar, how is it ?', additional_kwargs={}, response_metadata={}), AIMessage(content="A visually stunning and emotionally powerful space odyssey about humanity's survival and love across dimensions.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Im wat

In [40]:
response = llm.invoke(chat_values)

print(response.content)

*Joker* is a dark, character‑driven drama that strips away the usual comic‑book trappings to focus on Arthur Fleck’s descent into madness. The film’s gritty, 1970s‑inspired New York setting, its unsettling atmosphere, and Joaquin Phoenix’s raw, Oscar‑winning performance make it a haunting, thought‑provoking look at how society can shape—and break—a troubled individual. It’s intense, unsettling, and stays with you long after the credits roll.
